### TAO remote client - Classification

Transfer learning is the process of transferring learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. Train Adapt Optimize (TAO) Toolkit  is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

![image](https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png)

### Sample prediction for an Image Classification model
<img align="center" src="../example_images/sample_image_classification.jpg">

### The workflow in a nutshell

- Pulling datasets from cloud
- Getting a PTM from NGC
- Model Actions
    - Train (Normal/AutoML)
    - Evaluate
    - Prune, retrain
    - Export
    - Tao-Deploy
    - Inference on TAO, TRT
    - Delete experiments/dataset

### Table of contents

1. [Install TAO remote client](#head-1)
1. [FIXME's](#head-2)
1. [Login](#head-3)
1. [Create a cloud workspace](#head-2)
1. [Set dataset formats](#head-4)
1. [Create and pull train dataset](#head-5)
1. [Create and pull val dataset](#head-6)
1. [Create and pull test dataset](#head-7)
1. [List the created datasets](#head-8)
1. [Create experiment (via create-job)](#head-9)
1. [Assign train, eval datasets](#head-10)
1. [List experiments](#head-11)
1. [Assign PTM](#head-12)
1. [Set AutoML related configurations](#head-13)
1. [Train](#head-14)
1. [View hyperparameters that are enabled by default](#head-14.1)
1. [Evaluate](#head-15)
1. [Optimize: Prune, retrain and evaluate](#head-16)
1. [Export](#head-17)
1. [TRT Engine generation using TAO-Deploy](#head-18)
1. [TAO inference](#head-19)
1. [TRT inference](#head-20)
1. [Delete experiment](#head-21)
1. [Delete dataset](#head-22)

### Requirements
Please find the server requirements [here](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_setup.html#)

### Install TAO remote client <a class="anchor" id="head-1"></a>

In [ ]:
%%bash
# SKIP this step IF you have already installed the TAO-Client wheel.
pip3 install nvidia-tao-client

In [ ]:
%%bash
# View the version of the TAO-Client
tao --version

### Bash-based notebook
This notebook uses **Bash** (e.g. `%%bash` cells). State between cells is kept in `.tao_nb_state` in this directory. Ensure `tao login` has been run and `jq` is installed if you parse JSON output. When piping command output to `jq`, use **--output json** so the CLI emits JSON (default is text).

In [ ]:
%%bash
touch .tao_nb_state
source .tao_nb_state
which jq >/dev/null 2>&1 || { echo "Installing jq..."; sudo apt-get install -y jq 2>/dev/null || brew install jq; }
which yq >/dev/null 2>&1 || { echo "Installing yq..."; sudo wget -qO /usr/local/bin/yq https://github.com/mikefarah/yq/releases/latest/download/yq_linux_amd64 && sudo chmod +x /usr/local/bin/yq; }

In [ ]:
%%bash
# Restore state from previous run (re-run from top to reset)
source .tao_nb_state
echo "Current state: MODEL_NAME=${MODEL_NAME:-not set} WORKSPACE_ID=${WORKSPACE_ID:-not set}"

### To see the dataset folder structure required for the models supported in this notebook, visit the notebooks under dataset_prepare like for [this notebook](../dataset_prepare/classification.ipynb)

### FIXME's <a class="anchor" id="head-2"></a>

1. Assign a model_name in FIXME 1
1. (Optional) Enable AutoML if needed in FIXME 2
1. (Optional) To use AutoML: run get-automl-defaults (e.g. --base-experiment-id ID --action train --output @automl_defaults.json) then create-job with --automl-settings @automl_defaults.json
1. Assign a workdir in FIXME 4 for log file download
1. Assign the ip_address and port_number in FIXME 5 ([info](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_rest_api.html))
1. Assign the ngc_key variable in FIXME 6
1. Assign the ngc_org_name variable in FIXME 7
1. Set cloud storage details in FIXME 8
1. Assign path of datasets relative to the bucket in FIXME 9
1. Database backup/restore archive filename in FIXME 10

#### Choose a classification model

In [ ]:
%%bash
source .tao_nb_state
# Available models (#FIXME 1): classification_pyt - https://docs.nvidia.com/tao/tao-toolkit/text/image_classification.html
# FIXME 1: Set model name
export MODEL_NAME="${TAO_MODEL_NAME:-classification_pyt}"
echo "export MODEL_NAME=$MODEL_NAME" >> .tao_nb_state
echo "MODEL_NAME=$MODEL_NAME"

#### Toggle AutoML params
[AutoML documentation](https://docs.nvidia.com/tao/tao-toolkit/text/automl/automl.html#getting-started)

In [ ]:
%%bash
source .tao_nb_state
# FIXME 2: Set to true to enable AutoML (e.g. export TAO_AUTOML_ENABLED=true before running)
export AUTOML_ENABLED="${TAO_AUTOML_ENABLED:-false}"
# FIXME 3: One of bayesian/hyperband
export AUTOML_ALGORITHM="${TAO_AUTOML_ALGORITHM:-bayesian}"
echo "export AUTOML_ENABLED=$AUTOML_ENABLED" >> .tao_nb_state
echo "export AUTOML_ALGORITHM=$AUTOML_ALGORITHM" >> .tao_nb_state
echo "AUTOML_ENABLED=$AUTOML_ENABLED AUTOML_ALGORITHM=$AUTOML_ALGORITHM"

### Monitoring jobs in Bash
Run in a loop: `tao $MODEL_NAME get-job-metadata --job-id $JOB_ID --output json` and `tao $MODEL_NAME get-job-logs --job-id $JOB_ID` until status is Done or Error.

#### Example: poll job status

In [ ]:
%%bash
source .tao_nb_state
# Example: monitor a job until Done or Error (set JOB_ID or use JOB_ID_TRAIN)
JOB_ID="${JOB_ID:-$JOB_ID_TRAIN}"
[ -z "$JOB_ID" ] && echo "No JOB_ID set - skipping monitor example" && exit 0
while true; do
  STATUS=$(tao "$MODEL_NAME" get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  [ -z "$STATUS" ] && echo "Waiting for job..." && sleep 10 && continue
  echo "Status: $STATUS"
  tao "$MODEL_NAME" get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -5
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] || [ "$STATUS" = "Canceled" ] || [ "$STATUS" = "Paused" ] && break
  sleep 10
done

#### Login
Run `tao login` in a cell (or in your terminal). Credentials are stored in `~/.tao/config`.

#### Set API service's host information

In [ ]:
%%bash
source .tao_nb_state
# FIXME 4: Set TAO API environment variables (optional; often set by tao login)
# export TAO_BASE_URL=... export TAO_ORG=... export TAO_TOKEN=...

#### Set NGC Personal key for authentication and NGC org to access API services

In [ ]:
%%bash
source .tao_nb_state
# FIXME 6: Set your NGC key
export NGC_KEY="${NGC_KEY:-your_ngc_key}"
echo "export NGC_KEY=$NGC_KEY" >> .tao_nb_state
echo "NGC_KEY is set"

In [ ]:
%%bash
source .tao_nb_state
# FIXME 7: Set your NGC org
export NGC_ORG="${NGC_ORG:-nvstaging}"
echo "export NGC_ORG=$NGC_ORG" >> .tao_nb_state
echo "NGC_ORG=$NGC_ORG"

### Login <a class="anchor" id="head-3"></a>

In [ ]:
%%bash
source .tao_nb_state
tao login --ngc-org-name "${NGC_ORG:-nvstaging}" --ngc-key "${NGC_KEY:-your_ngc_key}" --enable-telemetry

### Get GPU details <a class="anchor" id="head-2"></a>

 Use --backend-type lepton --platform-id <key> (or --backend-type slurm --partition <name>) when you run create-job. Keys from get-gpu-types can be used as the platform ID.

In [ ]:
%%bash
# Optional: get GPU types for Slurm/Lepton backends
# tao get-gpu-types --output json | jq .

### Create cloud workspace
This workspace will be the place where your datasets reside and your results of TAO API jobs will be pushed to.

If you want to have different workspaces for dataset and experiment, duplocate the workspace creation part and adjust the metadata accordingly.

In [ ]:
%%bash
source .tao_nb_state
# FIXME 7/8: Set workspace cloud details (env or edit below)
export TAO_WORKSPACE_NAME="${TAO_WORKSPACE_NAME:-AWS workspace info}"
export TAO_WORKSPACE_CLOUD_TYPE="${TAO_WORKSPACE_CLOUD_TYPE:-aws}"
export TAO_WORKSPACE_CLOUD_REGION="${TAO_WORKSPACE_CLOUD_REGION:-us-west-1}"
export TAO_WORKSPACE_CLOUD_BUCKET_NAME="${TAO_WORKSPACE_CLOUD_BUCKET_NAME:-bucket_name}"
export TAO_WORKSPACE_CLOUD_ACCESS_KEY="${TAO_WORKSPACE_CLOUD_ACCESS_KEY:-access_key}"
export TAO_WORKSPACE_CLOUD_SECRET_KEY="${TAO_WORKSPACE_CLOUD_SECRET_KEY:-secret_key}"
echo "export TAO_WORKSPACE_NAME=$TAO_WORKSPACE_NAME" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_TYPE=$TAO_WORKSPACE_CLOUD_TYPE" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_REGION=$TAO_WORKSPACE_CLOUD_REGION" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_BUCKET_NAME=$TAO_WORKSPACE_CLOUD_BUCKET_NAME" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_ACCESS_KEY=$TAO_WORKSPACE_CLOUD_ACCESS_KEY" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_SECRET_KEY=$TAO_WORKSPACE_CLOUD_SECRET_KEY" >> .tao_nb_state
echo "export TAO_WORKSPACE_NAME=$TAO_WORKSPACE_NAME" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
# Create workspace (AWS example; use create-workspace-azure etc. for other clouds)
WORKSPACE_RESPONSE=$(tao $MODEL_NAME create-workspace-aws --name "AWS Workspace" \
  --cloud-region "$TAO_WORKSPACE_CLOUD_REGION" \
  --cloud-bucket-name "$TAO_WORKSPACE_CLOUD_BUCKET_NAME" \
  --access-key "$TAO_WORKSPACE_CLOUD_ACCESS_KEY" --secret-key "$TAO_WORKSPACE_CLOUD_SECRET_KEY" --output json)
WORKSPACE_ID=$(echo "$WORKSPACE_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$WORKSPACE_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "WORKSPACE_ID=$WORKSPACE_ID"
echo "export WORKSPACE_ID=$WORKSPACE_ID" >> .tao_nb_state

In [ ]:
%%bash
# Optional: Restore workspace from backup
# tao $MODEL_NAME restore-workspace --workspace-id $WORKSPACE_ID --backup-file-name mongodump.tar.gz

#### Set dataset URI (URI within cloud storage)

URIs should point to your dataset location in cloud storage, e.g. `aws://bucket-name/path/to/dataset`.

In [ ]:
%%bash
source .tao_nb_state
# FIXME 8: Paths relative to cloud bucket
export TRAIN_DATASET_URI="${TAO_TRAIN_DATASET_URI:-/data/classification_train}"
export EVAL_DATASET_URI="${TAO_EVAL_DATASET_URI:-/data/classification_val}"
export TEST_DATASET_URI="${TAO_TEST_DATASET_URI:-/data/classification_test}"
echo "export TRAIN_DATASET_URI=$TRAIN_DATASET_URI" >> .tao_nb_state
echo "export EVAL_DATASET_URI=$EVAL_DATASET_URI" >> .tao_nb_state
echo "export TEST_DATASET_URI=$TEST_DATASET_URI" >> .tao_nb_state
echo "TRAIN_DATASET_URI=$TRAIN_DATASET_URI"

In [ ]:
%%bash
source .tao_nb_state
# Dataset type/format for classification
export DS_TYPE=image_classification
export DS_FORMAT=$MODEL_NAME

### Assign PTM <a class="anchor" id="head-12"></a>

Search for PTM on NGC for the Classification model chosen

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME list-base-experiments --filter-param network_arch=$MODEL_NAME --output json | jq -r '.[] | "\(.name)\t\(.id)\t\(.network_arch)"'

In [ ]:
%%bash
source .tao_nb_state
# Select PTM by matching ngc_path suffix (like Python pretrained_map). Override PTM_NGC_PATH_SUFFIX or set SELECTED_PTM_ID to change.
export PTM_NGC_PATH_SUFFIX="${PTM_NGC_PATH_SUFFIX:-nvidia/tao/pretrained_fan_classification_imagenet:fan_hybrid_small}"
if [ -z "$SELECTED_PTM_ID" ]; then
  JSON=$(tao $MODEL_NAME list-base-experiments --filter-param network_arch=$MODEL_NAME --output json)
  if [ -n "$PTM_NGC_PATH_SUFFIX" ]; then
    export SELECTED_PTM_ID=$(echo "$JSON" | jq -r --arg s "$PTM_NGC_PATH_SUFFIX" '.[] | select(.ngc_path != null and (.ngc_path | endswith($s))) | .id' | head -1)
  fi
  if [ -z "$SELECTED_PTM_ID" ]; then
    export SELECTED_PTM_ID=$(echo "$JSON" | jq -r '.[0].id // empty')
  fi
  [ -n "$SELECTED_PTM_ID" ] && echo "export SELECTED_PTM_ID=$SELECTED_PTM_ID" >> .tao_nb_state
fi
echo "SELECTED_PTM_ID=$SELECTED_PTM_ID"

### Train <a class="anchor" id="head-16"></a>

#### View hyperparameters that are enabled for AutoML by default <a class="anchor" id="head-15"></a>

In [ ]:
%%bash
source .tao_nb_state
# View AutoML defaults (if AUTOML_ENABLED=true)
if [ "$AUTOML_ENABLED" = "true" ]; then
  tao $MODEL_NAME get-automl-defaults --base-experiment-id "$SELECTED_PTM_ID" --action train --output @automl_defaults.json
  jq . automl_defaults.json
fi

#### Set AutoML related configurations <a class="anchor" id="head-14.1"></a>
Refer to these hyper-links to see the parameters supported by each network and add more parameters if necessary in addition to the default automl enabled parameters: 

[Classification Pytorch](https://github.com/NVIDIA/tao_front_end_services/tree/main/api/specs_utils/specs/classification_pyt/classification_pyt%20-%20train.csv)

In [ ]:
%%bash
source .tao_nb_state
export ENCODE_KEY="${ENCODE_KEY:-tlt_encode}"
echo "export ENCODE_KEY=$ENCODE_KEY" >> .tao_nb_state
# For AutoML run get-automl-defaults then add --automl-settings @automl_defaults.json to create-job

#### Provide train specs

In [ ]:
%%bash
source .tao_nb_state
BASE_OPT=""
[ -n "$SELECTED_PTM_ID" ] && BASE_OPT="--base-experiment-id $SELECTED_PTM_ID"
tao $MODEL_NAME get-job-schema --defaults-only --action train $BASE_OPT --output @train_spec.yaml
head -80 train_spec.yaml

In [ ]:
%%bash
source .tao_nb_state
# Optional: edit train_spec.yaml with yq or sed, then copy to train_spec_filled.yaml
cp -f train_spec.yaml train_spec_filled.yaml
# Example: yq -i '.default.train.num_epochs = 80' train_spec_filled.yaml

#### Run train

In [ ]:
%%bash
source .tao_nb_state

if [ "$AUTOML_ENABLED" = "true" ]; then
  PARAMS=$(cat automl_defaults.json)

  # Algorithm-specific params
  if [ "$AUTOML_ALGORITHM" = "bayesian" ] || [ "$AUTOML_ALGORITHM" = "bfbo" ]; then
    ALGO_PARAMS='{"automl_max_recommendations": 20}'
  elif [ "$AUTOML_ALGORITHM" = "hyperband" ]; then
    ALGO_PARAMS='{"automl_max_epochs": 27, "automl_reduction_factor": 3, "epoch_multiplier": 1}'
  else
    ALGO_PARAMS='{}'
  fi

  jq -n --arg algo "$AUTOML_ALGORITHM" \
        --argjson params "$PARAMS" \
        --argjson algo_params "$ALGO_PARAMS" '{
    automl_enabled: true,
    automl_algorithm: $algo,
    automl_delete_intermediate_ckpt: false,
    override_automl_disabled_params: false,
    automl_hyperparameters: ($params | tostring),
    algorithm_specific_params: $algo_params,
    metric: "kpi"
  }' > automl_settings.json

  cat automl_settings.json
else
  echo "AutoML is disabled - training will use standard approach"
fi

In [ ]:
%%bash
source .tao_nb_state
AUTOML_OPT=""
[ -f automl_defaults.json ] && AUTOML_OPT="--automl-settings @automl_settings.json"
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action train --name ${MODEL_NAME}_training_job \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --base-experiment-id "$SELECTED_PTM_ID" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --eval-dataset-uri "$EVAL_DATASET_URI" --specs @train_spec_filled.yaml $AUTOML_OPT --output json)
JOB_ID_TRAIN=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRAIN=$JOB_ID_TRAIN"
echo "export JOB_ID_TRAIN=$JOB_ID_TRAIN" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRAIN"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

In [ ]:
%%bash
# To stop an AutoML job: stop the monitor cell, then run: tao $MODEL_NAME pause-job --job-id $JOB_ID_TRAIN

In [ ]:
%%bash
# To resume: tao $MODEL_NAME resume-job --job-id $JOB_ID_TRAIN --specs @train_spec_filled.yaml

### Publish model

#### Edit the method of choosing checkpoint from list of train checkpoint files

In [ ]:
%%bash
source .tao_nb_state
# List checkpoint files from the training job
tao $MODEL_NAME list-job-files --job-id "$JOB_ID_TRAIN" --output json | jq .

# Choose checkpoint method: latest_model, best_model, or from_epoch_number
tao $MODEL_NAME update-job --job-id "$JOB_ID_TRAIN" \
  --checkpoint-choose-method best_model

# # Or if you need a specific epoch:
# tao $MODEL_NAME update-job --job-id "$JOB_ID_TRAIN" \
#   --checkpoint-choose-method from_epoch_number \
#   --checkpoint-epoch-number 10

#### Push model to private ngc team registry

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME publish-model --job-id $JOB_ID_TRAIN --display-name "TAO $MODEL_NAME" --team "tao" --description "Train $MODEL_NAME"

#### Remove model from private ngc team registry

In [ ]:
%%bash
# tao $MODEL_NAME remove-published-model --job-id $JOB_ID --team <team>

### Evaluate <a class="anchor" id="head-15"></a>

#### Provide evaluate specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action evaluate --output @eval_spec.yaml
cp -f eval_spec.yaml eval_spec_filled.yaml

In [ ]:
%%bash
# Edit eval_spec_filled.yaml if needed (yq/sed)

#### Run evaluate

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action evaluate --name ${MODEL_NAME}_eval \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --eval-dataset-uri "$EVAL_DATASET_URI" --specs @eval_spec_filled.yaml --output json)
JOB_ID_EVAL=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EVAL=$JOB_ID_EVAL"
echo "export JOB_ID_EVAL=$JOB_ID_EVAL" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EVAL"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### Post-training quantization of the model <a class="anchor" id="head-19"></a>

#### Provide Quantization specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action quantize --output @quantize_spec.yaml
cp -f quantize_spec.yaml quantize_spec_filled.yaml

In [ ]:
%%bash
source .tao_nb_state
# Set float8 quantize layers in quantize spec
yq -i '.quantize.layers = [{"module_name": "Linear", "weights": {"dtype": "float8_e4m3fn"}, "activations": {"dtype": "float8_e4m3fn"}}, {"module_name": "Conv2d", "weights": {"dtype": "float8_e4m3fn"}, "activations": {"dtype": "float8_e4m3fn"}}]' quantize_spec_filled.yaml

#### Run Quantization

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action quantize --name ${MODEL_NAME}_quantize \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --specs @quantize_spec_filled.yaml --output json)
JOB_ID_QUANTIZE=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_QUANTIZE=$JOB_ID_QUANTIZE"
echo "export JOB_ID_QUANTIZE=$JOB_ID_QUANTIZE" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_QUANTIZE"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TAO inference <a class="anchor" id="head-19"></a>

#### Provide TAO inference specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action inference --output @tao_inference_spec.yaml
cp -f tao_inference_spec.yaml tao_inference_spec_filled.yaml

In [ ]:
%%bash
source .tao_nb_state
# Set is_quantized=true and quantize layers in inference spec
yq -i '.inference.is_quantized = true' tao_inference_spec_filled.yaml
yq -i '.quantize.layers = [{"module_name": "Linear", "weights": {"dtype": "float8_e4m3fn"}, "activations": {"dtype": "float8_e4m3fn"}}, {"module_name": "Conv2d", "weights": {"dtype": "float8_e4m3fn"}, "activations": {"dtype": "float8_e4m3fn"}}]' tao_inference_spec_filled.yaml

#### Run TAO inference

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_tao_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_QUANTIZE" --train-dataset-uri "$TRAIN_DATASET_URI" --eval-dataset-uri "$EVAL_DATASET_URI" \
  --specs @tao_inference_spec_filled.yaml --output json)
JOB_ID_TAO_INF=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TAO_INF=$JOB_ID_TAO_INF"
echo "export JOB_ID_TAO_INF=$JOB_ID_TAO_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TAO_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### Export <a class="anchor" id="head-17"></a>

#### Provide export specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action export --output @export_spec.yaml
cp -f export_spec.yaml export_spec_filled.yaml

In [ ]:
%%bash
# Edit export_spec_filled.yaml if needed

#### Run export

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action export --name ${MODEL_NAME}_export \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --specs @export_spec_filled.yaml --output json)
JOB_ID_EXPORT=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EXPORT=$JOB_ID_EXPORT"
echo "export JOB_ID_EXPORT=$JOB_ID_EXPORT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EXPORT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TRT Engine generation using TAO-Deploy <a class="anchor" id="head-18"></a>

#### Provide trt engine generation specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action gen_trt_engine --output @gen_trt_engine_spec.yaml
cp -f gen_trt_engine_spec.yaml gen_trt_engine_spec_filled.yaml

In [ ]:
%%bash
# Edit gen_trt_engine_spec_filled.yaml if needed

#### Run TRT Engine generation using TAO-Deploy

In [ ]:
%%bash
source .tao_nb_state
JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action gen_trt_engine --name ${MODEL_NAME}_gen_trt_engine \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_EXPORT" --specs @gen_trt_engine_spec_filled.yaml --output json)
JOB_ID_TRT=$(echo "$JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT=$JOB_ID_TRT"
echo "export JOB_ID_TRT=$JOB_ID_TRT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TRT inference <a class="anchor" id="head-20"></a>

#### Provide TRT inference specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --defaults-only --action inference --output @trt_inference_spec.yaml
cp -f trt_inference_spec.yaml trt_inference_spec_filled.yaml

In [ ]:
%%bash
# Edit tao_inference_spec_filled.yaml if needed

#### Run TRT inference

In [ ]:
%%bash
source .tao_nb_state
TRT_INF_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_trt_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRT" --inference-dataset-uri "$TEST_DATASET_URI" \
  --specs @trt_inference_spec_filled.yaml --output json)
JOB_ID_TRT_INF=$(echo "$TRT_INF_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$TRT_INF_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT_INF=$JOB_ID_TRT_INF"
echo "export JOB_ID_TRT_INF=$JOB_ID_TRT_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

In [ ]:
%%bash
# Optional: tao $MODEL_NAME backup-workspace --workspace-id $WORKSPACE_ID --backup-file-name mongodump.tar.gz

### Delete jobs <a class="anchor" id="head-22"></a>

In [ ]:
%%bash
source .tao_nb_state
for j in $JOB_ID_TRAIN $JOB_ID_EVAL $JOB_ID_QUANTIZE $JOB_ID_TAO_INF $JOB_ID_EXPORT $JOB_ID_TRT $JOB_ID_TRT_INF; do
  [ -n "$j" ] && tao $MODEL_NAME delete-job --job-id "$j" --confirm
done